In [ ]:
from google.colab import drive
drive.mount("/content/drive")


# Semantic-texture operational preflight

The first cell mounts Drive. Run all once in a fresh GPU runtime. This stable notebook clones the latest GitHub main branch into fresh `/content`, records the observed checkout revision, copies the sole Drive checkpoint input into `/content`, invokes only the repository bootstrap, and exports one zero-science quartet.

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from hashlib import sha256
import io
import json
import os
from pathlib import Path
import shutil
import stat
import subprocess
import sys
from uuid import uuid4
import zipfile

from google.colab import userdata

sys.tracebacklimit = 0
PROJECT_REPOSITORY_URL = "https://github.com/RICHAAARC/CEG-WM.git"
PROJECT_BRANCH = "main"
CHECKPOINT_RELATIVE_PATH = Path("MyDrive/CEG-WM/models/inspyrenet/ckpt_base.pth")
DELIVERY_COMPLETION_CHECKSUMS_FILENAME = "SHA256SUMS"

def _sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _copy_create_only(source: Path, destination: Path) -> None:
    source_stat = source.lstat()
    if not stat.S_ISREG(source_stat.st_mode) or source.name != "ckpt_base.pth" or destination.exists():
        raise RuntimeError("checkpoint copy boundary failed")
    destination.parent.mkdir(parents=True, exist_ok=False)
    with source.open("rb") as input_handle, destination.open("xb") as output_handle:
        shutil.copyfileobj(input_handle, output_handle)

def _persist_preclone_transport_failure(root: Path, run_id: str, blocked_class: str, *, drive_delivery_complete: bool, root_precreated: bool = False) -> None:
    if type(drive_delivery_complete) is not bool or type(root_precreated) is not bool or blocked_class not in {"environment_blocked", "resource_blocked", "implementation_blocked", "identity_blocked", "integrity_blocked"}:
        raise RuntimeError("pre-clone transport boundary failed")
    if root_precreated:
        root_stat = root.lstat()
        if not stat.S_ISDIR(root_stat.st_mode) or any(root.iterdir()):
            raise RuntimeError("pre-created transport root boundary failed")
    else:
        if root.exists():
            raise RuntimeError("pre-clone transport root already exists")
        root.mkdir(parents=True)
    result_name = "semantic_texture_operational_transport_result.json"
    archive_name = f"semantic_texture_transport_{run_id}.zip"
    receipt_name = "semantic_texture_operational_transport_receipt.json"
    result = {"aggregate": None, "blocked_class": blocked_class, "candidate_promoted": False, "drive_delivery_complete": drive_delivery_complete, "formal_tau_created": False, "profile_id": "semantic_texture_operational_preflight_transport", "run_id": run_id, "science_started": False, "scientific_claims_supported": False, "scientific_unit_count": 0, "status": "blocked", "transport_kind": "notebook_preclone_failure"}
    result_blob = (json.dumps(result, allow_nan=False, indent=2, sort_keys=True) + "\n").encode("utf-8")
    with (root / result_name).open("xb") as result_handle:
        result_handle.write(result_blob)
    with zipfile.ZipFile(root / archive_name, mode="x", compression=zipfile.ZIP_STORED) as archive:
        archive.writestr(result_name, result_blob)
    receipt = {"archive_filename": archive_name, "archive_sha256": _sha256_file(root / archive_name), "archive_size_bytes": (root / archive_name).stat().st_size, "blocked_class": blocked_class, "drive_delivery_complete": drive_delivery_complete, "profile_id": "semantic_texture_operational_preflight_transport", "result_filename": result_name, "result_sha256": sha256(result_blob).hexdigest(), "run_id": run_id, "status": "blocked"}
    receipt_blob = (json.dumps(receipt, allow_nan=False, indent=2, sort_keys=True) + "\n").encode("utf-8")
    with (root / receipt_name).open("xb") as receipt_handle:
        receipt_handle.write(receipt_blob)
    checksum_blob = f"{_sha256_file(root / result_name)}  {result_name}\n{_sha256_file(root / archive_name)}  {archive_name}\n{_sha256_file(root / receipt_name)}  {receipt_name}\n".encode("ascii")
    with (root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME).open("xb") as checksums_handle:
        checksums_handle.write(checksum_blob)

def _validate_delivery(artifact_root: Path, run_id: str) -> tuple[tuple[str, ...], dict[str, tuple[int, str]]]:
    operational_result = artifact_root / "semantic_texture_operational_result.json"
    transport_result = artifact_root / "semantic_texture_operational_transport_result.json"
    if operational_result.is_file() == transport_result.is_file():
        raise RuntimeError("exactly one operational or transport result is required")
    result_path = operational_result if operational_result.is_file() else transport_result
    archive_name = f"semantic_texture_operational_{run_id}.zip" if operational_result.is_file() else f"semantic_texture_transport_{run_id}.zip"
    receipt_name = "semantic_texture_operational_receipt.json" if operational_result.is_file() else "semantic_texture_operational_transport_receipt.json"
    names = (result_path.name, archive_name, receipt_name, DELIVERY_COMPLETION_CHECKSUMS_FILENAME)
    if {path.name for path in artifact_root.iterdir() if path.is_file()} != set(names):
        raise RuntimeError("delivery quartet is incomplete")
    result_blob = result_path.read_bytes()
    result = json.loads(result_blob)
    if result.get("status") != "blocked" or result.get("aggregate") is not None or result.get("science_started") is not False or result.get("scientific_unit_count") != 0 or result.get("candidate_promoted") is not False:
        raise RuntimeError("zero-science boundary drifted")
    archive_path = artifact_root / archive_name
    with zipfile.ZipFile(archive_path) as archive:
        if archive.namelist() != [result_path.name] or archive.read(result_path.name) != result_blob:
            raise RuntimeError("result-only archive boundary drifted")
    receipt_path = artifact_root / receipt_name
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    if receipt.get("result_sha256") != sha256(result_blob).hexdigest() or receipt.get("archive_sha256") != _sha256_file(archive_path):
        raise RuntimeError("receipt binding drifted")
    lines = (artifact_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME).read_text(encoding="ascii").splitlines()
    if [line.split("  ", 1)[1] for line in lines] != list(names[:3]):
        raise RuntimeError("completion roster drifted")
    for line in lines:
        digest, name = line.split("  ", 1)
        if _sha256_file(artifact_root / name) != digest:
            raise RuntimeError("completion digest drifted")
    blobs = b"".join((artifact_root / name).read_bytes() for name in names)
    if any(fragment in blobs for fragment in (b"HF_TOKEN", b"CEG_WM_ROOT_KEY", b"a red cube", b"/content/", b"traceback")):
        raise RuntimeError("private state crossed persistence boundary")
    return names, {name: ((artifact_root / name).stat().st_size, _sha256_file(artifact_root / name)) for name in names}


In [ ]:
mount_run_id = "semantic-texture-operational-mount-" + uuid4().hex
content_root = Path("/content")
local_parent = content_root / "ceg-wm-semantic-texture-operational"
mount_local_root = local_parent / mount_run_id
if mount_local_root.exists():
    raise RuntimeError("fresh local root is required; resume, migrate, and overwrite are forbidden")
mount_local_root.mkdir(parents=True)
exports_parent = Path("/content/drive/MyDrive/CEG-WM/semantic_texture_operational_preflight/exports")
try:
    exports_parent.mkdir(parents=True, exist_ok=True)
except OSError:
    _persist_preclone_transport_failure(mount_local_root / "transport-delivery", mount_run_id, "resource_blocked", drive_delivery_complete=False)
    raise RuntimeError("Drive export parent creation blocked") from None
for run_root_attempt in range(8):
    run_id = "semantic-texture-operational-" + uuid4().hex
    local_root = local_parent / run_id
    try:
        local_root.mkdir(parents=False, exist_ok=False)
    except FileExistsError:
        continue
    except OSError:
        _persist_preclone_transport_failure(mount_local_root / "transport-delivery", mount_run_id, "resource_blocked", drive_delivery_complete=False)
        raise RuntimeError("local run root creation blocked") from None
    drive_export_root = exports_parent / run_id
    try:
        drive_export_root.mkdir(parents=False, exist_ok=False)
    except FileExistsError:
        continue
    except OSError:
        _persist_preclone_transport_failure(local_root / "transport-delivery", run_id, "resource_blocked", drive_delivery_complete=False)
        raise RuntimeError("Drive run root creation blocked") from None
    break
else:
    _persist_preclone_transport_failure(mount_local_root / "transport-delivery", mount_run_id, "resource_blocked", drive_delivery_complete=False)
    raise RuntimeError("fresh run identity allocation blocked") from None
checkout_root = local_root / "repository"
execution_root = local_root / "execution"
checkpoint_path = local_root / "assets/inspyrenet/ckpt_base.pth"
try:
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch", PROJECT_REPOSITORY_URL, str(checkout_root)], check=True, capture_output=True)
    observed_repository_revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=checkout_root, check=True, capture_output=True, text=True).stdout.strip()
except Exception as error:
    blocked_class = "resource_blocked" if isinstance(error, (MemoryError, OSError)) else "environment_blocked" if isinstance(error, (subprocess.SubprocessError, RuntimeError)) else "implementation_blocked"
    _persist_preclone_transport_failure(drive_export_root, run_id, blocked_class, drive_delivery_complete=True, root_precreated=True)
    raise RuntimeError("pre-clone operational bootstrap blocked") from None
try:
    _copy_create_only(Path("/content/drive") / CHECKPOINT_RELATIVE_PATH, checkpoint_path)
    hf_token = userdata.get("HF_TOKEN")
    root_key = userdata.get("CEG_WM_ROOT_KEY")
    if not isinstance(hf_token, str) or not hf_token or not isinstance(root_key, str) or not root_key:
        raise RuntimeError("required Colab Secrets are unavailable")
except Exception as error:
    blocked_class = "resource_blocked" if isinstance(error, (MemoryError, OSError)) else "environment_blocked" if isinstance(error, (subprocess.SubprocessError, RuntimeError)) else "implementation_blocked"
    _persist_preclone_transport_failure(drive_export_root, run_id, blocked_class, drive_delivery_complete=True, root_precreated=True)
    raise RuntimeError("pre-clone operational bootstrap blocked") from None


In [ ]:
environment = os.environ.copy()
environment.update({"HF_TOKEN": hf_token, "CEG_WM_ROOT_KEY": root_key, "PYTHONDONTWRITEBYTECODE": "1"})
bootstrap_command = [sys.executable, str(checkout_root / "scripts/experiment_execution/semantic_texture_operational_preflight_bootstrap.py"), "--repository-root", str(checkout_root), "--checkpoint", str(checkpoint_path), "--execution-root", str(execution_root), "--entrypoint-args", "--execute", "--observed-repository-revision", observed_repository_revision, "--run-id", run_id, "--output-root", str(local_root / "operational-delivery")]
try:
    completed = subprocess.run(bootstrap_command, check=False, capture_output=True, env=environment)
except (MemoryError, OSError, subprocess.SubprocessError) as error:
    blocked_class = "resource_blocked" if isinstance(error, (MemoryError, OSError)) else "environment_blocked"
    del environment, hf_token, root_key
    _persist_preclone_transport_failure(drive_export_root, run_id, blocked_class, drive_delivery_complete=True, root_precreated=True)
    raise RuntimeError("operational bootstrap launch blocked") from None
del environment, hf_token, root_key
operational_root = local_root / "operational-delivery"
transport_root = execution_root.with_name(execution_root.name + ".transport")
artifact_root = operational_root if operational_root.is_dir() else transport_root
artifact_names, identities = _validate_delivery(artifact_root, run_id)
for name in artifact_names[:3]:
    source = artifact_root / name
    destination = drive_export_root / name
    with source.open("rb") as input_handle, destination.open("xb") as output_handle:
        shutil.copyfileobj(input_handle, output_handle)
    if (destination.stat().st_size, _sha256_file(destination)) != identities[name]:
        raise RuntimeError("Drive artifact identity drifted")
completion_blob = (artifact_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME).read_bytes()
pending = drive_export_root / ".completion.pending"
with pending.open("xb") as output_handle:
    output_handle.write(completion_blob)
    output_handle.flush()
    os.fsync(output_handle.fileno())
os.replace(pending, drive_export_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME)
if (drive_export_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME).read_bytes() != completion_blob:
    raise RuntimeError("completion marker changed during Drive delivery")
if completed.returncode != 0:
    raise RuntimeError("operational preflight blocked after complete Drive delivery") from None
